# Pingyu's NSTX ETG / KBM linear runs (132xxx)

First look at Pingyu's GENE runs under `/global/cfs/cdirs/m2116/TPED_GENE_simulations/`:
`NSTX_ETG`, `NSTX_KBM`, `NSTX_KBM_more`. He scanned 132xxx discharges with
**omt / omne scalings** of the profiles, and ran with **nx0 = 128** where our
scans use 64, so these are a reference for both the physics and the resolution.

Sections:
1. discover every run (one row per `parameters_NNNN`)
2. resolution table (nx0, nz0, nv0, nw0, nky0, box) per group
3. x0 radii sampled, per discharge and scaling
4. scaled profiles, read from the `iterdb_file` each run's parameters points at
5. γ and ω vs ky
6. mode identification from fingerprints (+ Epar) via the TPED registry

Reads go through TPED's GENE readers. **Read failures are expected** and are not
fixed here: each one is caught, recorded in `ERRORS`, and the run carries on
without that quantity. Section 7 tabulates them for the follow-up.

In [ ]:
import os, re, io, contextlib, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from TPED.projects.GENE_sim_reader.utils.find_GENE_files import GeneFileFinder as GFF
from TPED.projects.GENE_sim_analysis.src.sim_assembler import GeneSimulationPlus
from TPED.projects.utils.read_iterdb import read_iterdb

# The readers log a full traceback per failed file; those land in ERRORS below instead.
logging.getLogger("TPED").setLevel(logging.CRITICAL)
pd.set_option("display.max_rows", 200, "display.max_colwidth", 80)

## Inputs

In [ ]:
PARENT = "/global/cfs/cdirs/m2116/TPED_GENE_simulations"
ROOTS = {g: os.path.join(PARENT, g) for g in ("NSTX_ETG", "NSTX_KBM", "NSTX_KBM_more")}
MAX_DEPTH = 6          # how deep GeneFileFinder looks for parameters files under each root
OUR_NX0 = 64           # our scans' nx0, for the resolution comparison

RES_COLS = ["nx0", "nky0", "nz0", "nv0", "nw0", "lx", "lv", "lw", "n_spec"]

## 1. Discover runs

One `GeneSimulationPlus` per parameters file. Built one at a time rather than with
`GeneSimulationAssembler(root)`, which would drop a whole group on the first bad file.

`shot` and the **omt / omne scale factors** are parsed from the paths in the
parameters file (`iterdb_file`, then `geomdir` / `geomfile`, then the run directory),
in the IFS naming `omt0p7_omne1p1` (`p` = decimal point; `1.2` also accepted).
A run whose paths carry no scaling gets `NaN` there; check `scale_source` if the
parse looks wrong.

In [ ]:
ERRORS = []   # every caught failure: group, run, stage, message

def log_error(group, where, stage, exc):
    msg = exc if isinstance(exc, str) else f"{type(exc).__name__}: {exc}"
    # run path out of the message, so section 7 groups identical failures
    msg = re.sub(r"'[^']*parameters[^']*'", "<run>", msg.strip().splitlines()[-1]).split(" load_errors:")[0]
    ERRORS.append({"group": group, "where": where, "stage": stage, "error": msg[:300]})

NUM = r"(\d+(?:[p.]\d+)?)"
SCALE_RE = {"omt_scale": re.compile(rf"omt_?{NUM}", re.I),
            "omne_scale": re.compile(rf"omne?_?{NUM}", re.I)}
SHOT_RE = re.compile(r"(?<!\d)(1[23]\d{4})(?!\d)")

def par(ds, key):
    if key not in ds:
        return None
    v = ds[key].values
    v = v.item() if v.size == 1 else v
    return v.decode() if isinstance(v, bytes) else v

def resolve(path, sim_dir):
    # GENE reads relative paths from the run dir, which the diagdir does not record:
    # try the scan dir and its parent, else keep the path as written.
    if not path or os.path.isabs(path):
        return path
    for base in (sim_dir, os.path.dirname(sim_dir)):
        cand = os.path.normpath(os.path.join(base, path))
        if os.path.exists(cand):
            return cand
    return path

def parse_path_tags(*paths):
    out = {"shot": None, "omt_scale": np.nan, "omne_scale": np.nan, "scale_source": None}
    for p in filter(None, paths):
        if out["shot"] is None and (m := SHOT_RE.findall(p)):
            out["shot"] = int(m[-1])
        if out["scale_source"] is None:
            hits = {k: rx.findall(p) for k, rx in SCALE_RE.items()}
            if all(hits.values()):
                for k, h in hits.items():
                    out[k] = float(h[-1].replace("p", "."))
                out["scale_source"] = p
    return out

PAR_KEYS = ["x0", "kymin", "kx_center", *RES_COLS, "beta", "coll", "magn_geometry",
            "omt_e", "omn_e", "omt_i", "omn_i", "temp_e", "dens_e"]

sims, rows = {}, []
for group, root in ROOTS.items():
    try:
        paths = sorted(GFF(root).find_files(filetype="parameters", max_depth=MAX_DEPTH))
    except Exception as exc:
        log_error(group, root, "discover", exc)
        continue
    for fp in paths:
        try:
            sim = GeneSimulationPlus(fp)
            ds = sim.parameters_xarray
        except Exception as exc:
            log_error(group, fp, "parameters", exc)
            continue
        iterdb = resolve(par(ds, "iterdb_file"), sim.directory)
        geomdir, geomfile = par(ds, "geomdir"), par(ds, "geomfile")
        key = (group, fp)
        sims[key] = sim
        rows.append({"group": group, "run": fp,
                     **parse_path_tags(iterdb, geomdir, geomfile, sim.directory),
                     **{k: par(ds, k) for k in PAR_KEYS},
                     "iterdb_file": iterdb, "geomdir": geomdir, "geomfile": geomfile,
                     "suffix": sim.suffix, "directory": sim.directory})

runs = pd.DataFrame(rows)
num = [c for c in PAR_KEYS if c != "magn_geometry"]
runs[num] = runs[num].apply(pd.to_numeric, errors="coerce")
runs["scaling"] = [f"omt{t:g}_omne{n:g}" if pd.notna(t) else "unscaled?"
                   for t, n in zip(runs.omt_scale, runs.omne_scale)]
runs.groupby("group").agg(n_runs=("run", "size"), shots=("shot", lambda s: sorted(s.dropna().unique())),
                          n_scalings=("scaling", "nunique"))

In [ ]:
runs.head(10)

## 2. Resolution

Every distinct grid used, with the run count behind it. `nx0 != 64` flags where his
grid differs from ours.

In [ ]:
res = (runs.groupby(["group", *RES_COLS], dropna=False).size().rename("n_runs")
       .reset_index())
res["nx0_vs_ours"] = np.where(res.nx0 == OUR_NX0, "same", f"differs from {OUR_NX0}")
res

## 3. x0 radii sampled

In [ ]:
x0_table = (runs.groupby(["group", "shot", "scaling"], dropna=False)
            .agg(x0=("x0", lambda s: sorted(s.dropna().round(4).unique())),
                 kymin=("kymin", lambda s: sorted(s.dropna().round(3).unique())),
                 n_runs=("run", "size")))
x0_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.5 + 0.45 * runs.groupby(["group", "shot"]).ngroups))
labels = []
for i, ((group, shot), d) in enumerate(runs.groupby(["group", "shot"], dropna=False)):
    ax.scatter(d.x0, np.full(len(d), i), c=d.omt_scale, cmap="viridis", vmin=runs.omt_scale.min(),
               vmax=runs.omt_scale.max(), s=40, edgecolor="k", lw=0.3)
    labels.append(f"{group} | {shot}")
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel(r"x0 ($\rho_{tor}$)")
ax.set_title("x0 sampled (colour = omt scale)")
fig.colorbar(ax.collections[0], ax=ax, label="omt scale")
fig.tight_layout()

## 4. Scaled profiles

Each distinct `iterdb_file` the runs point at, read with TPED's `read_iterdb`.
One row per discharge, lines per scaling, dashed lines at the x0 values sampled.
A missing or unreadable iterdb is recorded in `ERRORS` and skipped.

In [ ]:
PROF_KEYS = {"Te": "TE", "Ti": "TI", "ne": "NE"}   # iterdb quantity names

profiles = {}   # iterdb path -> {Te: (rho, values), ...}
for path, d in runs.dropna(subset=["iterdb_file"]).groupby("iterdb_file"):
    try:
        with contextlib.redirect_stdout(io.StringIO()):   # read_iterdb prints its point count
            rhot, prof, _units = read_iterdb(path)
        profiles[path] = {lab: (np.asarray(rhot[k]), np.asarray(prof[k]))
                          for lab, k in PROF_KEYS.items() if k in prof}
    except Exception as exc:
        log_error(d.group.iloc[0], path, "iterdb", exc)

prof_runs = runs[runs.iterdb_file.isin(profiles)]
shots = sorted(prof_runs.shot.dropna().unique())
fig, axes = plt.subplots(max(len(shots), 1), len(PROF_KEYS), figsize=(4 * len(PROF_KEYS), 3 * max(len(shots), 1)),
                         squeeze=False, sharex=True)
for r, shot in enumerate(shots):
    d_shot = prof_runs[prof_runs.shot == shot]
    for (scaling, path), d in d_shot.groupby(["scaling", "iterdb_file"]):
        for c, lab in enumerate(PROF_KEYS):
            if lab in profiles[path]:
                axes[r, c].plot(*profiles[path][lab], label=scaling, lw=1)
    for c, lab in enumerate(PROF_KEYS):
        for x0 in d_shot.x0.dropna().unique():
            axes[r, c].axvline(x0, ls="--", c="grey", lw=0.6)
        axes[r, c].set_title(f"{shot}  {lab}")
    axes[r, 0].legend(fontsize=6)
for ax in axes[-1]:
    ax.set_xlabel(r"$\rho_{tor}$")
fig.tight_layout()

## 5. γ and ω vs ky

`omega` file per run, in that run's GENE normalisation: check his reference
quantities (`Lref`, `Bref`, `Tref`) against ours before overlaying the two scans.
Points with γ ≤ 0 or NaN are unconverged or stable.

In [ ]:
om_rows = []
for (group, fp), sim in sims.items():
    sim.return_sim_xarray(["omega"])
    tree = sim.tree
    if "omega" not in tree.children:
        log_error(group, fp, "omega", tree.attrs["load_errors"].get("omega", "no omega node"))
        continue
    ds = tree["omega"].ds
    om_rows.append({"run": fp, "gamma": float(ds["gamma"].values), "omega": float(ds["omega"].values)})

runs = runs.drop(columns=["gamma", "omega"], errors="ignore").merge(pd.DataFrame(om_rows, columns=["run", "gamma", "omega"]), on="run", how="left")
runs.groupby("group").agg(n_runs=("run", "size"), with_omega=("gamma", "count"),
                          growing=("gamma", lambda g: int((g > 0).sum())))

In [ ]:
CURVE = ["shot", "scaling", "x0"]

for group, dg in runs.dropna(subset=["gamma"]).groupby("group"):
    shots_g = sorted(dg.shot.dropna().unique()) or [None]
    fig, axes = plt.subplots(2, len(shots_g), figsize=(5 * len(shots_g), 7), squeeze=False, sharex=True)
    for c, shot in enumerate(shots_g):
        ds_ = dg if shot is None else dg[dg.shot == shot]
        for (_, scaling, x0), d in ds_.groupby(CURVE, dropna=False):
            d = d.sort_values("kymin")
            lab = f"{scaling}, x0={x0:.3f}"
            axes[0, c].plot(d.kymin, d.gamma, "o-", ms=3, lw=1, label=lab)
            axes[1, c].plot(d.kymin, d.omega, "o-", ms=3, lw=1, label=lab)
        axes[0, c].set_title(f"{group}  {shot}")
        axes[1, c].set_xlabel(r"$k_y\rho_s$")
        axes[0, c].legend(fontsize=6)
    axes[0, 0].set_ylabel(r"$\gamma$")
    axes[1, 0].set_ylabel(r"$\omega$")
    for ax in axes.flat:
        ax.axhline(0, c="k", lw=0.5)
        if dg.kymin.max() / max(dg.kymin.min(), 1e-12) > 20:
            ax.set_xscale("log")
    fig.tight_layout()

## 6. Mode identification (fingerprints + Epar)

Registry analyses per run: `fingerprints` (nrg), `epar` (field + geometry + omega),
then `mode_identifier` (MTM → ETG → KBM → ITG/TEM, first passing check wins;
thresholds in `TPED/projects/GENE_sim_analysis/src/analyses/mode_identifier.py`).
Each step is tried separately, so a run missing `field` still gets its fingerprint
ratios. Only growing runs (γ > 0) are classified. Cross-species ratios need both
electrons and ions — an adiabatic-ion ETG run will fail `mode_identifier` and show
up in `ERRORS`.

In [ ]:
FP_COLS = ["D_e-Chi_e", "Chi_i-Chi_e", "Q_EM_e-|Q_ES|_tot", "D-chi_tot_e"]
EP_COLS = ["Epar_norm", "parity_phi", "parity_apar"]

def scalar(ds, k):
    return float(np.asarray(ds[k].values).squeeze()) if k in ds else np.nan

id_rows = []
for (group, fp), sim in sims.items():
    if not runs.loc[runs.run == fp, "gamma"].gt(0).any():
        continue
    row = {"run": fp}
    for name, cols in (("fingerprints", FP_COLS), ("epar", EP_COLS), ("mode_identifier", ["mode_ID"])):
        try:
            node = getattr(sim.analysis, name)().ds
            for k in cols:
                row[k] = str(node[k].values) if k == "mode_ID" else scalar(node, k)
        except Exception as exc:
            log_error(group, fp, name, exc)
    id_rows.append(row)

ids = pd.DataFrame(id_rows, columns=["run", *FP_COLS, *EP_COLS, "mode_ID"])
runs = runs.drop(columns=[*FP_COLS, *EP_COLS, "mode_ID"], errors="ignore").merge(ids, on="run", how="left")
runs["mode_ID"] = runs["mode_ID"].fillna(pd.Series(np.where(runs.gamma > 0, "failed", "not growing"), index=runs.index))
pd.crosstab([runs.group, runs.shot], runs.mode_ID, margins=True)

In [ ]:
pd.crosstab([runs.group, runs.shot, runs.scaling], runs.mode_ID)

In [ ]:
MODE_C = {"ETG": "tab:blue", "KBM": "tab:red", "MTM": "tab:green", "ITG_TEM": "tab:orange",
          "unidentified": "tab:grey", "failed": "k", "not growing": "lightgrey"}
groups = sorted(runs.group.unique())
fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 4), squeeze=False)
for ax, group in zip(axes[0], groups):
    d = runs[(runs.group == group) & runs.gamma.notna()]
    for mode, dm in d.groupby("mode_ID"):
        ax.scatter(dm.kymin, dm.gamma, s=18, c=MODE_C.get(mode, "m"), label=f"{mode} ({len(dm)})")
    ax.set_title(group)
    ax.set_xlabel(r"$k_y\rho_s$")
    ax.axhline(0, c="k", lw=0.5)
    ax.legend(fontsize=7)
axes[0, 0].set_ylabel(r"$\gamma$")
fig.tight_layout()

In [ ]:
runs[["group", "shot", "scaling", "x0", "kymin", "gamma", "omega", *FP_COLS, *EP_COLS, "mode_ID"]].sort_values(["group", "shot", "scaling", "x0", "kymin"])

## 7. Read errors

Everything caught above, grouped by stage and message. The fixes come later; this
is the list.

In [ ]:
errors = pd.DataFrame(ERRORS, columns=["group", "where", "stage", "error"])
errors.groupby(["stage", "error"]).agg(n=("where", "size"), groups=("group", lambda g: sorted(set(g))),
                                       example=("where", "first"))